In [ ]:
!pip install git+https://github.com/Koziev/rusyllab

In [ ]:
import re
import pandas as pd
import numpy as np
import rusyllab
from scipy import stats
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
import tqdm
import os
import matplotlib.pyplot as plt
import seaborn as sns
import json
from collections import Counter
import tqdm

import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from scipy.stats import shapiro, wilcoxon

# Расчет метрик

В работе для оценки качества автоматической синтаксической разметки используются следующие меьрики: UAS, LAS, LA, SLA

**UAS** (Unlabeled Attachment Score) рассчитывает процент слов, для которых был верно определен родительский узел (количество слов с верно распознанным родителем деленное на общее количество слов).

**LAS** (Labeled Attachment Score) в свою очередь рассчитывает процент слов, для которых парсер верно определил и родительский узел, и тип связи (количество верно обработанных слов, поделенное на общее количество слов).

**LA** (Labeled Accuracy) рассчитывает процент слов, для которых парсер верно определил синтаксический тег.

**SLA** (Sentence-Level Accuracy) рассчитывает процент предложений, для которых все теги (и теги синтаксической роли, и теги связи) были обозначены верно.

In [ ]:
def UAS(sentence, corr_sentence):
  sentence = sentence.split()
  try:
    corr_sentence = corr_sentence.split()
    if len(sentence) == len(corr_sentence):
      counter = 0
      for i in range(len(sentence)):
        try:
          if sentence[i].split('|')[1] == corr_sentence[i].split('|')[1]:
            counter += 1
        except:
          counter += 0
      return counter / len(sentence)

    else:
      return 0
  except:
    return 0

In [ ]:
def labeled_accuracy(sentence, corr_sentence):
  sentence = sentence.split()
  try:
    corr_sentence = corr_sentence.split()
    if len(sentence) == len(corr_sentence):
      counter = 0
      for i in range(len(sentence)):
        try:
          if sentence[i].split('|')[2] == corr_sentence[i].split('|')[2]:
            counter += 1
        except:
          counter += 0
      return counter / len(sentence)

    else:
      return 0
  except:
    return 0

In [ ]:
def LAS(sentence, corr_sentence):
  sentence = sentence.split()
  try:
    corr_sentence = corr_sentence.split()
    if len(sentence) == len(corr_sentence):
      counter = 0
      for i in range(len(sentence)):
        try:
          if sentence[i].split('|')[1] == corr_sentence[i].split('|')[1] and sentence[i].split('|')[2] == corr_sentence[i].split('|')[2]:
            counter += 1
        except:
          counter += 0
      return counter / len(sentence)

    else:
      return 0
  except:
    return 0

In [ ]:
def roots(text):
  result = re.sub(r"(\d+)(?=\|ROOT)", '0', text)
  return result

In [ ]:
def count_metrics(file_name):
  sample = pd.read_csv(f'{file_name}.csv')

  speakers = []
  fixed_sentence = []
  preliminary_syntax = []
  for i in range(len(sample)):
    synt =  roots(sample['Изначальный синтаксис'].iloc[i])
    preliminary_syntax.append(synt)
    for_split = sample['Исправленная запись'].iloc[i].split(':')
    speaker = for_split[0]
    fixed_sentence.append(for_split[1])
    speakers.append(speaker.replace('*', ''))

  sample['Говорящий'] = speakers
  sample['Изначальный синтаксис'] = preliminary_syntax
  sample['Исправленная запись'] = fixed_sentence

  syntax = []
  lengths = []
  syntax_lengths = []
  uases = []
  lases = []
  labeled_accuracies = []
  syllables = []
  words = []
  for i in range(len(sample)):
    phrase = len(re.sub(r'[^\w\s]', '', sample['Исправленная запись'].iloc[i]).split())
    words.append(phrase)
    sentence = sample['Изначальный синтаксис'].iloc[i]
    syntax_lengths.append(len(sample['Изначальный синтаксис'].iloc[i].split()))
    corr_sentence = sample['Исправленный синтаксис'].iloc[i]
    syllable = len(rusyllab.split_words(sample['Исправленная запись'].iloc[i].replace('.', '').split()))
    syllables.append(syllable)
    try:
      lengths.append(len(corr_sentence.split()))
    except:
      lengths.append(0)

    uases.append(UAS(sentence, corr_sentence))
    lases.append(LAS(sentence, corr_sentence))
    try:
      labeled_accuracies.append(labeled_accuracy(sentence, corr_sentence))
    except:
      labeled_accuracies.append(0)

  sample['Тип речи'] = np.where(sample['Говорящий'].isin(['CHI', 'PAR1']), 'Child', 'Adult')
  sample['Слов в фразе'] = words
  sample['Слогов в фразе'] = syllables
  sample['SLA'] = np.where(sample['Изначальный синтаксис'] == sample['Исправленный синтаксис'], 1, 0)
  sample['Длина исправленного разбора'] = lengths
  sample['Длина изначального разбора'] = syntax_lengths
  sample['UAS'] = uases
  sample['LAS'] = lases
  sample['labeled_accuracy'] = labeled_accuracies
  speakers = set(speakers)
  sample.to_csv(f'updated_sample_{file_name}.csv')
  return sample, speakers

## MLU


Вдобавок к метрикам оценки качества синтаксического парсинга рассчитывалась MLU для слов и слогов с библиотеки rusyllab, которая позволяет делить слова на слоги на основе правил фонетики и фонологии.

MLU (Mean Length of Utterance) -- стандартная метрика оценки сложности детской речи, которая представляет собой среднее количество слов слогов или морфем во фразе

In [ ]:
def final_results(sample):
  speech_types = ['Child', 'Adult']
  results = {'words': {}, 'sentences': {}, 'mean_UAS': {}, 'mean_LAS': {}, 'mean_LA': {},
             'SLA': {}, 'MLU_words': {}, 'MLU_syllables': {},
             'precision': {}, 'recall': {}}


  for speech_type in speech_types:
    data = sample[sample['Тип речи'] == speech_type]
    sentences = list(data['Исправленная запись'])
    words = []
    for sentence in sentences:
      sentence = re.sub(r'[^\w\s]', '', sentence).split()
      for word in sentence:
        words.append(word)
    results['words'][speech_type] = len(words)
    results['sentences'][speech_type] = len(sentences)
    results['mean_UAS'][speech_type] = float(round(data['UAS'].mean(), 4))
    results['mean_LAS'][speech_type] = float(round(data['LAS'].mean(), 4))
    results['mean_LA'][speech_type] = float(round(data['labeled_accuracy'].mean(), 4))
    results['SLA'][speech_type] = float(round((data['SLA'].sum() / len(data['SLA'])), 4))
    results['MLU_words'][speech_type] = float(round(data['Слов в фразе'].mean(), 4))
    results['MLU_syllables'][speech_type] = float(round(data['Слогов в фразе'].mean(), 4))
    results['precision'][speech_type] = precision_score(sample['Исправленный синтаксис'], sample['Изначальный синтаксис'], average='micro')
    results['recall'][speech_type] = recall_score(sample['Исправленный синтаксис'], sample['Изначальный синтаксис'], average='micro')

  return results

# Обработка данных

In [ ]:
# функция для подсчета метрик
def final_function(file_name):
  final_result = {}
  sample, speakers = count_metrics(file_name)
  results = final_results(sample)
  final_result[file_name] = results
  return final_result

In [ ]:
# обрабатываем все файлы
data = []

directory_path = '/content/'

for filename in tqdm.tqdm(os.listdir(directory_path)):
  file_path = os.path.join(directory_path, filename)
  if os.path.isfile(file_path):
    if filename.startswith('s'):
      filename = filename.split('.')[0]
      final_result = final_function(filename)

      data.append(final_result)

In [ ]:
# считаем средние метрики для ребенка и для взрослых
rows = []

for item in data:
    for file_id, metrics in item.items():
        for speaker in ['Child', 'Adult']:

            row = {
                'file_id': file_id,
                'speaker': speaker,

                'words': metrics['words'][speaker],
                'sentences': metrics['sentences'][speaker],

                'mean_UAS': metrics['mean_UAS'][speaker],
                'mean_LAS': metrics['mean_LAS'][speaker],
                'mean_LA': metrics['mean_LA'][speaker],

                'SLA': metrics['SLA'][speaker],

                'MLU_words': metrics['MLU_words'][speaker],
                'MLU_syllables': metrics['MLU_syllables'][speaker],

                'precision': metrics['precision'][speaker],
                'recall': metrics['recall'][speaker],
            }

            rows.append(row)

df = pd.DataFrame(rows)

df = df.replace({np.nan: None})

df.head()
df.to_csv('final_metrics.csv')

In [ ]:
# Подсчитываем количество типов ошибок для каждого файлы, также считаем процент ошибок в файле

directory_path = '/content/'

full_data = {
    'error_types': {},
    'error_subtypes': {},
    'corrections': {},
    'error_rate': {}
}


def process_subset(df):

    error_types = []
    error_subtypes = []
    corrections_all = []

    errors_df = df.dropna(subset=['Тип ошибки']).reset_index(drop=True)

    for i in range(len(errors_df)):

        errors = str(
            errors_df['Тип ошибки'].iloc[i]
        ).split(', ')

        error_types.extend(errors)

        subtypes = str(
            errors_df['Подтип ошибки'].iloc[i]
        ).split(', ')

        error_subtypes.extend(subtypes)

        corr_value = errors_df['Исправления'].iloc[i]

        if pd.notna(corr_value):

            corrections = str(corr_value).split(', ')
            corrections_all.extend(corrections)

    return {
        'error_types': Counter(error_types),
        'error_subtypes': Counter(error_subtypes),
        'corrections': Counter(corrections_all)
    }


for filename in tqdm.tqdm(os.listdir(directory_path)):

    file_path = os.path.join(directory_path, filename)

    if (
        os.path.isfile(file_path)
        and filename.startswith('updated_sample_')
    ):

        data = pd.read_csv(file_path)

        file_id = filename.split('.')[0].split('_')[2]

        print(file_id)

        subsets = {
            'General': data,
            'Child': data[data['Тип речи'] == 'Child'],
            'Adult': data[data['Тип речи'] == 'Adult']
        }

        full_data['error_rate'][file_id] = {}
        full_data['error_types'][file_id] = {}
        full_data['error_subtypes'][file_id] = {}
        full_data['corrections'][file_id] = {}

        for group_name, subset in subsets.items():

            n_errors = subset['Тип ошибки'].notna().sum()

            full_data['error_rate'][file_id][group_name] = {
                'number_of_errors': int(n_errors),
                'number_of_replics': len(subset),
                'error_rate': (
                    float(round(n_errors / len(subset), 4))
                    if len(subset) > 0 else None
                )
            }

            processed = process_subset(subset)

            full_data['error_types'][file_id][group_name] = processed['error_types']

            full_data['error_subtypes'][file_id][group_name] = processed['error_subtypes']

            full_data['corrections'][file_id][group_name] = processed['corrections']

In [ ]:
error_data = full_data['error_rate']
rows = []

for file_id, groups in error_data.items():

    for group_name, values in groups.items():

        rows.append({
            'file_id': file_id,
            'group': group_name,

            'number_of_errors': values['number_of_errors'],
            'number_of_replics': values['number_of_replics'],
            'error_rate': values['error_rate']
        })

error_df = pd.DataFrame(rows)

error_df.to_csv('error_rate.csv')

In [ ]:
error_types = full_data['error_types']
error_subtypes = full_data['error_subtypes']
corrections = full_data['corrections']

In [ ]:
# заносим статистику по типам и подтипам ошибок в общий файл

rows = []
for file_id, groups in error_types.items():
  for group_name, values in groups.items():

        rows.append({
            'file_id': file_id,
            'group': group_name,

            'ошибка в определении роли': values['ошибка в определении роли'],
            'ошибка в определении вершины': values['ошибка в определении вершины'],
            'ошибка в определении связи': values['ошибка в определении связи'],
            'ошибка распознавания': values['ошибка распознавания'],
            'пропуск': values['пропуск'],
            'общее количество ошибок': int(error_df[(error_df['file_id'] == file_id) & (error_df['group'] == group_name)]['number_of_errors'])
        })

types_df = pd.DataFrame(rows)

types_df.to_csv('types_numbers.csv')

In [ ]:
# счита6ем проценты разных типов и подтипов ошибок в файлах

error_cols = [
    'ошибка в определении роли',
    'ошибка в определении вершины',
    'ошибка в определении связи',
    'ошибка распознавания',
    'пропуск'
]


percent_df = types_df.copy()

percent_df[error_cols] = (
    percent_df[error_cols]
    .div(percent_df['общее количество ошибок'], axis=0)
    * 100
).round(2)

percent_df.to_csv('types_percents.csv')

In [ ]:
# считаем количество подтипов ошибок и сохраняем для каждого файлы топ-10 (тк для ошибок в определении синтаксической роли подтипов очень много)
rows = []


for file_id, groups in error_subtypes.items():

    for group_name, counter_obj in groups.items():

        top_10 = counter_obj.most_common(10)

        for tag, count in top_10:

            rows.append({
                'file_id': file_id,
                'group': group_name,
                'tag': tag,
                'count': count
            })

top10_df = pd.DataFrame(rows)

top10_df['error_class'] = np.where(top10_df['tag'].str.islower() == True, 'punct', 'tag')
top10_df.to_csv('error_subtypes_numbers.csv')

In [ ]:
# считаем частотность для подтипов ошибок в файлах

grouped = (
    top10_df
    .groupby(['file_id', 'group', 'error_class'])
    .apply(
        lambda x: x.assign(
            percent_within_class=(
                x['count'] / x['count'].sum() * 100
            ).round(2)
        )
    )
    .reset_index(drop=True)
)

grouped.head()
grouped.to_csv('error_subtypes_percents.csv')

In [ ]:
# смотрим на неверно определенные синтаксические роли, берем самые интересные нам

results = {'vocative': [],
           'discourse': [],
           'nsubj': [],
           'obj': [],
           'flat': [],
           'root': []}


tags = list(results.keys())

for tag in tqdm.tqdm(tags):
  for correction in corrections:
    if ('>') in correction:
      words = correction.split('>')
      try:
        incorrect = words[0].strip()
        correction = words[1].strip()
      except:
        print(words)
    if ('<') in correction:
      words = correction.split('<')
      try:
        incorrect = words[0].strip()
        correction = words[1].strip()
      except:
        print(words)

    if correction == tag:
      results[tag].append(correction)
  results[tag] = Counter(results[tag]).most_common()

In [ ]:
rows = []

for gold_tag, replacements in results.items():

    total = sum(count for _, count in replacements)

    for predicted_tag, count in replacements:

        rows.append({
            'gold': gold_tag,
            'predicted': predicted_tag,
            'count': count,
            'percent_within_gold': round(count / total * 100, 2)
        })

confusion_df = pd.DataFrame(rows)

confusion_df.to_csv('changes_df.csv')

# Статистический анализ

После проведения фильтрации высказываний на детские и взрослые мы рассчитывали среднее значение метрик, а также значения MLU в словах и слогах для взрослых и ребенка. В качестве финального этапа работы с данными все числовые показатели были сохранены в датафрейм с указанием номера файла.


Проверяли данные на нормальность с помощью **теста Шапиро-Уилка**, проверяли значимость различий между полученными значениями метрик для детской речи и речи, обращенной к детям с помощью **теста Уилкоксона**. Также мы проверяли значения MLU и полученных метрик у детей и взрослых на корреляцию посредством **подсчета коэффициента корреляции Спирмена**.


In [ ]:
data = pd.read_csv('final_metrics.csv').drop('Unnamed: 0', axis=1)

In [ ]:
data.head(5)

In [ ]:
data_adults = data[data['speaker'] == 'Adult']
data_child = data[data['speaker'] == 'Child']

In [ ]:
# средние знаения метрик по датасету

gen_data = data.groupby('speaker')[[
    'mean_UAS',
    'mean_LAS',
    'mean_LA',
    'SLA',
    'MLU_words',
    'MLU_syllables',
    'precision',
    'recall'
]].agg(['mean', 'std'])

gen_data.to_csv('general_data.csv')
gen_data

In [ ]:
# список метрик

metrics = ['mean_UAS',
    'mean_LAS',
    'mean_LA',
    'SLA',
    'MLU_words',
    'MLU_syllables',
    'precision',
    'recall']

In [ ]:
# сводная таблица

pivot = data.pivot(
    index='file_id',
    columns='speaker',
    values=['mean_UAS',
    'mean_LAS',
    'mean_LA',
    'SLA',
    'MLU_words',
    'MLU_syllables',
    'precision',
    'recall']
)

pivot.to_csv('pivot_table.csv')

In [ ]:
stat_metrics = ['mean_UAS',
    'mean_LAS',
    'mean_LA',
    'SLA',
    'precision',
    'recall']

In [ ]:
# статистические тесты

results = []

for metric in stat_metrics:
    child = pivot[metric]['Child']
    adult = pivot[metric]['Adult']

    mask = (~child.isna()) & (~adult.isna())

    child = child[mask]
    adult = adult[mask]

    diff = child - adult

    if np.allclose(diff, 0):
        print(f'{metric}: identical values, skipped')
        continue

    shapiro_stat, shapiro_p = shapiro(diff)
    wilcoxon_stat, wilcoxon_p = wilcoxon(
        child,
        adult,
        method='exact'
    )

    results.append({
        'metric': metric,

        'shapiro_W': round(shapiro_stat, 4),
        'shapiro_p': round(shapiro_p, 4),

        'wilcoxon_W': round(wilcoxon_stat, 4),
        'wilcoxon_p': round(wilcoxon_p, 4),

        'child_mean': round(child.mean(), 4),
        'adult_mean': round(adult.mean(), 4)
    })

wilcoxon = pd.DataFrame(results)

print(wilcoxon)

wilcoxon.to_csv('wilcoxon_test.csv')

In [ ]:
# корреляция для MLU и метрик

data_adults = data[data['speaker'] == 'Adult']
data_child = data[data['speaker'] == 'Child']

rows = []

for metric in stat_metrics:

    corr_child, p_child = spearmanr(
        data_child['MLU_words'],
        data_child[metric]
    )

    rows.append({
        'metric': metric,
        'group': 'Child',
        'spearman_r': round(corr_child, 4),
        'p_value': round(p_child, 4)
    })

    corr_adult, p_adult = spearmanr(
        data_adults['MLU_words'],
        data_adults[metric]
    )

    rows.append({
        'metric': metric,
        'group': 'Adult',
        'spearman_r': round(corr_adult, 4),
        'p_value': round(p_adult, 4)
    })

mlu_correlations = pd.DataFrame(rows)

mlu_correlations.to_csv('mlu_correlations.csv', index=False)

mlu_correlations

In [ ]:
data_adults = data[data['speaker'] == 'Adult']
data_child = data[data['speaker'] == 'Child']

In [ ]:
# корреляция для возраста и метрик
data_adults = data[data['speaker'] == 'Adult']
data_child = data[data['speaker'] == 'Child']

rows = []

for metric in stat_metrics:
    corr_child, p_child = spearmanr(
        data_child['child_age'],
        data_child[metric]
    )

    rows.append({
        'metric': metric,
        'group': 'Child',
        'spearman_r': round(corr_child, 4),
        'p_value': round(p_child, 4)
    })

    corr_adult, p_adult = spearmanr(
        data_adults['child_age'],
        data_adults[metric]
    )

    rows.append({
        'metric': metric,
        'group': 'Adult',
        'spearman_r': round(corr_adult, 4),
        'p_value': round(p_adult, 4)
    })

age_correlations = pd.DataFrame(rows)

age_correlations.to_csv(
    'age_correlations.csv',
    index=False
)

In [ ]:
df = pd.read_csv('types_percents.csv')